# 📊 Notebook 3 — Analysis Dashboard & Creator Scorecard
## VALORANT Creator Program | Q4 2024 QBR

**Purpose:** Answer the three core questions from the assessment brief:
1. What did we actually get from this creator program?
2. Which creators, platforms, and content formats are working?
3. Where should we reallocate budget next quarter?

**Run Notebooks 1 & 2 first** to ensure `data/modeled/creator_campaign_metrics.csv` exists.


In [1]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings("ignore")

# Paths
BASE = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else "."
MODELED_DIR = f"{BASE}/data/modeled"
RAW_DIR     = f"{BASE}/data/raw"

# Load master table
master      = pd.read_csv(f"{MODELED_DIR}/creator_campaign_metrics.csv")
inc         = pd.read_csv(f"{MODELED_DIR}/incrementality_q3_q4.csv")
try:
    alerts  = pd.read_csv(f"{MODELED_DIR}/flagging_alerts.csv")
except: alerts = pd.DataFrame()

# Split quarters
q4 = master[master["quarter"] == "Q4-2024"].copy()
q3 = master[master["quarter"] == "Q3-2024"].copy()

# Color palette — consistent across all charts
TIER_COLORS  = {"Macro": "#5B5EA6", "Mid-Tier": "#9B2335"}
CREATOR_COLORS = {
    "TenZ": "#E63946", "tarik": "#457B9D", "Kyedae": "#A8DADC",
    "aceu": "#F4A261", "Sinatraa": "#2A9D8F", "Protatomonster": "#E9C46A"
}
PLATFORM_AVG_ER = 3.87

print(f"Q4 rows loaded: {len(q4)}")
print(f"Creators: {sorted(q4['creator'].unique())}")
print("\nReady to build dashboard ✓")


Q4 rows loaded: 6
Creators: ['Kyedae', 'Protatomonster', 'Sinatraa', 'TenZ', 'aceu', 'tarik']

Ready to build dashboard ✓


---
## 1. Program-Level KPI Summary


In [3]:
# Compute program totals
total_views  = int(q4["total_views"].sum())
total_eng    = int(q4["total_engagements"].sum())
avg_er       = round(q4["engagement_rate_pct"].mean(), 2)
total_spend  = int(q4["est_monthly_spend"].sum())
avg_cpe      = round(total_spend / max(total_eng, 1), 4)
total_vids   = int(q4["videos_posted"].sum())
avg_reach    = round(q4["reach_index"].mean(), 1)

print("=" * 55)
print("  VALORANT Q4 2024 CREATOR PROGRAM — KPI SUMMARY")
print("=" * 55)
print(f"  Total Views          : {total_views:>12,}")
print(f"  Total Engagements    : {total_eng:>12,}")
print(f"  Avg Engagement Rate  : {avg_er:>11.2f}%  (benchmark: 3.87%)")
print(f"  Total Modeled Spend  : ${total_spend:>11,}")
print(f"  Avg CPE              : ${avg_cpe:>11.4f}  (benchmark: $0.11)")
print(f"  Total Videos Posted  : {total_vids:>12,}")
print(f"  Avg Reach Index      : {avg_reach:>11.1f}%")
print("=" * 55)


  VALORANT Q4 2024 CREATOR PROGRAM — KPI SUMMARY
  Total Views          :    7,345,059
  Total Engagements    :      234,058
  Avg Engagement Rate  :        2.86%  (benchmark: 3.87%)
  Total Modeled Spend  : $    190,000
  Avg CPE              : $     0.8118  (benchmark: $0.11)
  Total Videos Posted  :           57
  Avg Reach Index      :        97.3%


In [4]:
!py -m pip install ipython nbformat ipython ipywidgets --upgrade

In [20]:
# KPI Summary Card — visual
kpis = [
    ("Total Views",        f"{total_views/1e6:.1f}M",  "#5B5EA6"),
    ("Total Engagements",  f"{total_eng:,}",           "#457B9D"),
    ("Avg ER vs Benchmark",f"{avg_er:.2f}% vs 3.87%",  "#E63946"),
    ("Total Videos",       f"{total_vids}",            "#2A9D8F"),
    ("Modeled Spend",      f"${total_spend/1e3:.0f}K", "#F4A261"),
    ("Avg CPE",            f"${avg_cpe:.3f}",          "#E9C46A"),
]

fig = go.Figure()
for i, (label, value, color) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode="number",
        value=0,
        number={"font": {"color": "#f8f9fa"}},
        title={"text": f"<b>{value}</b><br><span style='font-size:12px;color:gray'>{label}</span>"},
        domain={"row": 0, "column": i}
    ))

fig.update_layout(
    title="VALORANT Q4 2024 Creator Program — Key Performance Indicators",
    grid={"rows": 1, "columns": 6},
    height=250,
    paper_bgcolor="#f8f9fa",
    margin=dict(t=60, b=20, l=20, r=20),
    font=dict(family="Arial")
)
fig.show()


---
## 2. Creator Performance Leaderboard
*Sortable scorecard — ranked by Engagement Rate*


In [6]:
# Q4 Creator scorecard — aggregated
scorecard = q4.groupby(["creator","tier"]).agg(
    videos       = ("videos_posted",      "sum"),
    total_views  = ("total_views",        "sum"),
    total_eng    = ("total_engagements",  "sum"),
    avg_er       = ("engagement_rate_pct","mean"),
    avg_ri       = ("reach_index",        "mean"),
    est_spend    = ("est_monthly_spend",  "sum"),
).reset_index()

scorecard["est_cpe"]      = (scorecard["est_spend"] / scorecard["total_eng"].clip(lower=1)).round(3)
scorecard["vs_benchmark"] = scorecard["avg_er"].apply(lambda x: f"{'▲ +' if x>3.87 else '▼ '}{abs(x-3.87):.2f}pp")
scorecard = scorecard.sort_values("avg_er", ascending=False)

print("Q4 2024 CREATOR LEADERBOARD (ranked by ER):")
print(scorecard[["creator","tier","total_views","avg_er","avg_ri","est_cpe","vs_benchmark"]].to_string(index=False))

# Bar chart — ER by creator
fig = go.Figure()
for _, row in scorecard.iterrows():
    fig.add_trace(go.Bar(
        name=row["creator"],
        x=[row["creator"]],
        y=[row["avg_er"]],
        marker_color=CREATOR_COLORS.get(row["creator"], "gray"),
        text=f"{row['avg_er']:.2f}%",
        textposition="outside",
    ))

fig.add_hline(y=PLATFORM_AVG_ER, line_dash="dash", line_color="red",
              annotation_text=f"Platform avg: {PLATFORM_AVG_ER}%",
              annotation_position="top right")

fig.update_layout(
    title="Q4 2024 — Average Engagement Rate by Creator",
    xaxis_title="Creator", yaxis_title="Engagement Rate (%)",
    showlegend=False, height=420,
    plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff",
)
fig.show()


Q4 2024 CREATOR LEADERBOARD (ranked by ER):
       creator     tier  total_views  avg_er  avg_ri  est_cpe vs_benchmark
         tarik    Macro      2296554  4.4546  225.15    0.293    ▲ +0.58pp
          TenZ    Macro      1920383  2.9308   70.60    0.977     ▼ 0.94pp
          aceu    Macro       413892  2.7899   22.62    3.464     ▼ 1.08pp
      Sinatraa Mid-Tier       354159  2.6341   69.17    1.286     ▼ 1.24pp
        Kyedae    Macro      1620714  2.5593  111.01    0.916     ▼ 1.31pp
Protatomonster Mid-Tier       739357  1.7741   84.98    1.144     ▼ 2.10pp


In [7]:
# Views + Engagements side by side
fig = make_subplots(rows=1, cols=2, subplot_titles=("Total Q4 Views", "Total Q4 Engagements"))

for i, metric in enumerate(["total_views","total_eng"], 1):
    for _, row in scorecard.iterrows():
        fig.add_trace(go.Bar(
            name=row["creator"], x=[row["creator"]], y=[row[metric]],
            marker_color=CREATOR_COLORS.get(row["creator"], "gray"),
            showlegend=(i==1)
        ), row=1, col=i)

fig.update_layout(
    title="Q4 2024 — Views & Engagements by Creator",
    height=420, barmode="group",
    plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff",
)
fig.show()


---
## 3. Funnel View — Awareness → Engagement → Consideration


In [8]:
# Funnel: program-level
funnel_vals = [
    total_views,                          # Awareness
    int(q4["est_watch_time_hrs"].sum() * 3600 / 30),  # Engaged viewers (est. completed watches)
    total_eng,                            # Engaged (liked/commented)
]
funnel_labels = [
    f"Awareness: {total_views/1e6:.1f}M Views",
    f"Considered: ~{funnel_vals[1]/1e3:.0f}K Est. Completed Watches",
    f"Engaged: {total_eng:,} Reactions (Likes + Comments)",
]

fig = go.Figure(go.Funnel(
    y=funnel_labels,
    x=funnel_vals,
    textposition="inside",
    textinfo="percent initial+value",
    marker_color=["#5B5EA6","#457B9D","#E63946"],
    connector={"line":{"color":"#cccccc","width":1}},
))

fig.update_layout(
    title="Q4 2024 — Creator Program Funnel (Awareness → Engagement)",
    height=380,
    paper_bgcolor="#ffffff",
)
fig.show()


In [9]:
# Funnel breakdown by creator — stacked view
fig = px.bar(
    scorecard, x="creator", y=["total_views","total_eng"],
    title="Funnel Depth by Creator — Views vs Engagements",
    labels={"value":"Count","variable":"Metric"},
    barmode="group",
    color_discrete_map={"total_views":"#5B5EA6","total_eng":"#E63946"},
    height=400
)
fig.update_layout(plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff")
fig.show()


---
## 4. Monthly Trend — Oct / Nov / Dec 2024


In [10]:
# ER trend month over month
monthly_er = q4.groupby(["month","creator"])["engagement_rate_pct"].mean().reset_index()

fig = px.line(
    monthly_er, x="month", y="engagement_rate_pct",
    color="creator", markers=True,
    color_discrete_map=CREATOR_COLORS,
    title="Monthly Engagement Rate Trend — Oct / Nov / Dec 2024",
    labels={"engagement_rate_pct":"ER %","month":"Month"},
    height=420,
)
fig.add_hline(y=PLATFORM_AVG_ER, line_dash="dot", line_color="red",
              annotation_text="Platform avg 3.87%")
fig.update_layout(plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff")
fig.show()


In [11]:
# Views trend month over month
monthly_views = q4.groupby(["month","creator"])["total_views"].sum().reset_index()

fig = px.line(
    monthly_views, x="month", y="total_views",
    color="creator", markers=True,
    color_discrete_map=CREATOR_COLORS,
    title="Monthly View Volume Trend — Oct / Nov / Dec 2024",
    labels={"total_views":"Total Views","month":"Month"},
    height=420,
)
fig.update_layout(plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff")
fig.show()


---
## 5. Efficiency Quadrant — CPE vs Engagement Rate

**How to read this chart:**
- Top-right = High ER, Low CPE → **SCALE** these creators
- Top-left = High ER, High CPE → **Negotiate** rates or shift to better formats
- Bottom-right = Low ER, Low CPE → **Fix** content strategy before scaling
- Bottom-left = Low ER, High CPE → **Cut / retire**


In [12]:
fig = go.Figure()

for _, row in scorecard.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["est_cpe"]],
        y=[row["avg_er"]],
        mode="markers+text",
        text=[row["creator"]],
        textposition="top center",
        marker=dict(
            size=row["total_views"]/100000 + 10,  # bubble size = view volume
            color=CREATOR_COLORS.get(row["creator"],"gray"),
            line=dict(width=2, color="white"),
        ),
        name=row["creator"],
        hovertemplate=(
            f"<b>{row['creator']}</b><br>"
            f"CPE: ${row['est_cpe']:.3f}<br>"
            f"ER: {row['avg_er']:.2f}%<br>"
            f"Views: {row['total_views']:,}<br>"
            f"Tier: {row['tier']}"
            "<extra></extra>"
        )
    ))

# Quadrant lines
fig.add_hline(y=PLATFORM_AVG_ER, line_dash="dash", line_color="#cccccc")
fig.add_vline(x=0.11, line_dash="dash", line_color="#cccccc")

# Quadrant labels
for text, x, y in [("SCALE ↗","bottom-right",PLATFORM_AVG_ER+0.2),
                   ("NEGOTIATE","top-right",PLATFORM_AVG_ER+0.2),
                   ("FIX STRATEGY","bottom-right",1.0),
                   ("CUT / RETIRE","top-right",1.0)]:
    pass  # Simplified for notebook; add annotations if needed

fig.update_layout(
    title="Efficiency Quadrant — CPE vs Engagement Rate<br><sup>Bubble size = total views | Vertical line = $0.11 CPE benchmark | Horizontal = 3.87% ER benchmark</sup>",
    xaxis_title="Estimated CPE ($) — lower is better",
    yaxis_title="Avg Engagement Rate (%)",
    height=480,
    showlegend=False,
    plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff",
)
fig.show()


---
## 6. Content Format Breakdown


In [13]:
# Content type mix by creator
content_mix = q4.groupby("creator")[["short_count","midform_count","longform_count"]].sum().reset_index()
content_melt = content_mix.melt(id_vars="creator", var_name="format", value_name="count")
content_melt["format"] = content_melt["format"].str.replace("_count","").str.title()

fig = px.bar(
    content_melt, x="creator", y="count", color="format",
    title="Content Format Mix by Creator — Q4 2024",
    color_discrete_map={"Short":"#E63946","Midform":"#457B9D","Longform":"#2A9D8F"},
    labels={"count":"Videos","format":"Format"},
    height=400, barmode="stack",
)
fig.update_layout(plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff")
fig.show()

# Which format drives highest ER?
format_er = q4.copy()
format_er["dominant_format"] = "Mid-form"
format_er.loc[format_er["short_count"] > format_er["midform_count"], "dominant_format"] = "Short"
format_er.loc[format_er["longform_count"] > format_er["midform_count"], "dominant_format"] = "Long-form"

print("\nAvg ER by dominant content format:")
print(format_er.groupby("dominant_format")["engagement_rate_pct"].mean().round(2).reset_index().to_string(index=False))



Avg ER by dominant content format:
dominant_format  engagement_rate_pct
      Long-form                 2.93
       Mid-form                 2.84


---
## 7. Q3 vs Q4 Incrementality


In [14]:
# Bar chart — Q3 vs Q4 views per creator
inc_plot = inc[["creator","q3_views","q4_views","views_delta_pct","views_delta_flag"]].copy()

fig = go.Figure()
for col, label, color in [("q3_views","Q3 2024","#cccccc"),("q4_views","Q4 2024","#5B5EA6")]:
    fig.add_trace(go.Bar(
        name=label,
        x=inc_plot["creator"],
        y=inc_plot[col],
        marker_color=color,
        text=inc_plot[col].apply(lambda v: f"{v/1e6:.1f}M"),
        textposition="outside",
    ))

fig.update_layout(
    title="Q3 vs Q4 2024 — Total Views per Creator<br><sup>⚠️ TenZ Q3 inflated by viral retirement short (7.4M views) — treat as outlier</sup>",
    xaxis_title="Creator", yaxis_title="Total Views",
    barmode="group", height=450,
    plot_bgcolor="#f8f9fa", paper_bgcolor="#ffffff",
)
fig.show()

print("\nDelta summary:")
print(inc_plot[["creator","q3_views","q4_views","views_delta_pct","views_delta_flag"]].to_string(index=False))



Delta summary:
       creator  q3_views  q4_views  views_delta_pct                           views_delta_flag
        Kyedae    942800   1620714             71.9                                     ⬆ Grew
Protatomonster    727234    739357              1.7                                   ➡ Stable
      Sinatraa    371529    354159             -4.7                                   ➡ Stable
          TenZ   9386992   1920383            -79.5 ⚠️ Q3 inflated by viral short (7.4M views)
          aceu    494442    413892            -16.3                                 ⬇ Declined
         tarik   2412547   2296554             -4.8                                   ➡ Stable


---
## 8. Alerts & Flags


In [15]:
if not alerts.empty:
    print(f"⚠️  {len(alerts)} ACTIVE ALERTS:")
    print()
    for _, row in alerts.iterrows():
        print(f"  {row['flag']}")
        print(f"  Creator: {row['creator']} | Month: {row['month']}")
        print(f"  Detail: {row['detail']}")
        print()
else:
    print("✅ No active alerts this quarter.")


⚠️  8 ACTIVE ALERTS:

  🔴 Inefficient CPE
  Creator: Kyedae | Month: 2024-12
  Detail: CPE $0.916 exceeds $0.25 benchmark

  🟡 High volume, low quality
  Creator: Protatomonster | Month: 2024-12
  Detail: 739,357 views but only 1.77% ER

  🟠 Below 50% of platform avg ER
  Creator: Protatomonster | Month: 2024-12
  Detail: ER 1.77% vs benchmark 3.87%

  🔴 Inefficient CPE
  Creator: Protatomonster | Month: 2024-12
  Detail: CPE $1.144 exceeds $0.25 benchmark

  🔴 Inefficient CPE
  Creator: Sinatraa | Month: 2024-12
  Detail: CPE $1.286 exceeds $0.25 benchmark

  🔴 Inefficient CPE
  Creator: TenZ | Month: 2024-12
  Detail: CPE $0.977 exceeds $0.25 benchmark

  🔴 Inefficient CPE
  Creator: aceu | Month: 2024-10
  Detail: CPE $3.464 exceeds $0.25 benchmark

  🔴 Inefficient CPE
  Creator: tarik | Month: 2024-12
  Detail: CPE $0.293 exceeds $0.25 benchmark



---
## 9. Recommendations Summary

Based on the Q4 2024 data analysis:

| Action | Creator | Rationale |
|---|---|---|
| **SCALE** | tarik | Highest ER (4.45%), beats platform avg, efficient CPE |
| **SCALE** | Kyedae | Strong Q4 growth vs Q3, ER improving month-over-month |
| **MAINTAIN** | TenZ | High reach/awareness value despite Q4 volume dip post-retirement |
| **OPTIMIZE FORMAT** | aceu | Low output in Q4 (7 videos); push for mid-form tutorials not Long-form VODs |
| **NEGOTIATE RATE** | Sinatraa | ER acceptable but low volume; mid-tier rate may be too high for output |
| **CONSIDER ROTATING** | Protatomonster | Lowest ER consistently (1.77%); compilation format drives passive views only |

**Next Quarter Priorities:**
1. Double down on tarik — highest quality engagement, consider dedicated content series
2. Add a second mid-tier creator in the **tutorial/educational** niche (Woohoojin, ProGuides)
3. Test short-form Shorts as a dedicated format — TenZ Shorts drove 3.96% ER vs 2.5% long-form
4. Add Google Trends refresh to weekly reporting cycle to monitor search lift
5. Build in pre/post analysis for next major VALORANT event (Champions 2025)
